# PeriPy Elastic Deformation Demo

Companion to `peripy_fracture_demo.ipynb`. Same plate geometry and
material, but we load the plate **below its critical stretch** so no
bonds break — the response stays purely elastic.

**Problem:** a 1.0 m × 0.5 m 2-D plate pulled in tension by small
prescribed displacements on the top and bottom edges. Because the load
stays in the elastic regime, we can validate PeriPy by comparing the
measured axial strain against the analytical prediction `u_y(y) = ε · y`.

**What this notebook does**
1. Generates the same triangular mesh as the fracture demo.
2. Sets the peak edge displacement to ~25 % of the bond-failure threshold.
3. Enables **dynamic-relaxation damping** so the plate settles into a
   quasi-static equilibrium instead of ringing forever.
4. Runs the peridynamic simulation — no pre-crack, no critical-stretch
   violations.
5. Plots the `u_y` field and fits `du_y/dy` at the midplane, comparing the
   PD result to linear elasticity (`σ = E · ε`).

## 1. Setup

Same environment as the fracture demo — PeriPy + meshio + an OpenCL
runtime. See `README.md` for installation notes.

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt

import meshio
from peripy.model import Model
from peripy.integrators import EulerCromerCL
print('PeriPy is available.')

## 2. Geometry and mesh

Structured 81 × 41 triangular mesh of the plate, with boundary `line`
elements so PeriPy's mesh loader can identify the perimeter nodes.

In [ ]:
L_x, L_y = 1.0, 0.5
nx, ny = 81, 41
dx = L_x / (nx - 1)
dy = L_y / (ny - 1)

xs = np.linspace(0.0, L_x, nx)
ys = np.linspace(0.0, L_y, ny)
xx, yy = np.meshgrid(xs, ys)
points = np.column_stack([xx.ravel(), yy.ravel(), np.zeros(nx * ny)])
nnodes = points.shape[0]

tris = []
for j in range(ny - 1):
    for i in range(nx - 1):
        n00 = j * nx + i
        n10 = j * nx + i + 1
        n01 = (j + 1) * nx + i
        n11 = (j + 1) * nx + i + 1
        tris.append([n00, n10, n11])
        tris.append([n00, n11, n01])
tris = np.asarray(tris, dtype=np.int64)

lines = []
for i in range(nx - 1):
    lines.append([i, i + 1])
for i in range(nx - 1):
    lines.append([(ny - 1) * nx + i, (ny - 1) * nx + i + 1])
for j in range(ny - 1):
    lines.append([j * nx, (j + 1) * nx])
for j in range(ny - 1):
    lines.append([j * nx + (nx - 1), (j + 1) * nx + (nx - 1)])
lines = np.asarray(lines, dtype=np.int64)

mesh_path = pathlib.Path('plate_elastic.msh')
meshio.write_points_cells(
    str(mesh_path),
    points=points,
    cells=[('line', lines), ('triangle', tris)],
    file_format='gmsh22',
    binary=False,
)
print(f'Wrote {mesh_path.resolve()}')
print(f'  {nnodes} nodes | {len(tris)} triangles | {len(lines)} boundary lines')

## 3. Material + PD parameters and elastic load level

With `critical_stretch = 5·10⁻⁴` and `L_y = 0.5 m`, the critical edge
displacement on one side is `L_y/2 · s_0 = 1.25·10⁻⁴ m`. We load only to
`U_MAX = 3·10⁻⁵ m` — 25 % of the failure threshold — so the plate stays
firmly in its elastic regime.

In [ ]:
# Material: brittle, glass-like (same as fracture demo for consistency)
E = 72.0e9          # Pa
rho = 2440.0        # kg/m^3
thickness = 0.01    # m

# PD discretisation
horizon = 3.015 * dx
critical_stretch = 5.0e-4
bond_stiffness = (9.0 * E) / (np.pi * thickness * horizon**3)
density = np.full(nnodes, rho, dtype=np.float64)

# Elastic load level
U_MAX = 3.0e-5
eps_applied = 2.0 * U_MAX / L_y

print(f'E                = {E:.2e} Pa')
print(f'horizon          = {horizon:.4e} m')
print(f'bond_stiffness   = {bond_stiffness:.4e} N/m^5')
print(f'critical_stretch = {critical_stretch:.2e}')
print(f'U_MAX / edge     = {U_MAX:.2e} m  ({U_MAX / (L_y/2*critical_stretch):.1%} of critical)')
print(f'applied strain   = {eps_applied:.3e}  ({eps_applied/critical_stretch:.1%} of critical stretch)')

## 4. Boundary conditions

No pre-crack this time. Top and bottom clamp layers are pulled apart with
prescribed displacement; every interior node is free.

In [ ]:
BC_LAYER = 3.0 * dx  # thickness of the pulled clamp layer

def is_displacement_boundary(x):
    bnd = [None, None, None]
    if x[1] > L_y - BC_LAYER:
        bnd[1] = 1
    elif x[1] < BC_LAYER:
        bnd[1] = -1
    return bnd

## 5. Build the model with dynamic-relaxation damping

For a *quasi-static* elastic problem we want the plate to settle into
equilibrium, not oscillate. `EulerCromerCL` accepts a `damping` parameter
(units kg/(m³·s)) that adds a viscous drag proportional to velocity to the
equation of motion. A useful rule of thumb is

$$\eta \sim 2 \, \rho \, c_p / \delta,$$

where `c_p = √(E/ρ)` is the P-wave speed and `δ` is the horizon. That
critically damps the longest-wavelength mode in the plate.

In [ ]:
dt = 1.0e-8
wave_speed = np.sqrt(E / rho)
damping = 2.0 * rho * wave_speed / horizon
print(f'wave speed = {wave_speed:.2e} m/s')
print(f'damping    = {damping:.3e} kg/(m^3 s)')

integrator = EulerCromerCL(damping=damping, dt=dt)

model = Model(
    mesh_file=str(mesh_path),
    integrator=integrator,
    horizon=horizon,
    critical_stretch=critical_stretch,
    bond_stiffness=bond_stiffness,
    dimensions=2,
    density=density,
    is_displacement_boundary=is_displacement_boundary,
    # no initial_crack - intact elastic plate
)
print(f'Nodes: {model.nnodes}  |  max neighbours/node: {model.max_neighbours}')

## 6. Load schedule: ramp, then hold

We ramp the cumulative boundary displacement from 0 to `U_MAX` over the
first half of the run, then **hold** at `U_MAX` for the second half. The
hold phase lets dynamic-relaxation damping bleed off any residual
kinetic energy so the final state is an elastostatic equilibrium.

In [ ]:
n_steps = 4000
hold_start = n_steps // 2

ramp = np.linspace(0.0, U_MAX, hold_start)
hold = np.full(n_steps - hold_start, U_MAX)
displacement_bc_magnitudes = np.concatenate([ramp, hold])

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(displacement_bc_magnitudes * 1e6)
ax.axvline(hold_start, color='k', lw=0.5, ls='--')
ax.set_xlabel('step'); ax.set_ylabel('edge disp [μm]')
ax.set_title('Load schedule (cumulative edge displacement)')
ax.grid(alpha=0.3); plt.show()

## 7. Run

Because the load is sub-critical, expect **zero damaged nodes** at the end.

In [ ]:
u, damage, connectivity, force, ud, data = model.simulate(
    steps=n_steps,
    displacement_bc_magnitudes=displacement_bc_magnitudes,
    write=500,
)
print(f'u.shape       = {u.shape}')
print(f'damage range  = [{damage.min():.3e}, {damage.max():.3e}]')
print(f'damaged nodes = {int((damage > 0.0).sum())} / {damage.size}')

## 8. Validation against linear elasticity

In the midplane (away from boundary artifacts) the response should be
uniaxial tension: `u_y(y) = ε · (y − L_y/2)`. We fit the slope and compare
to the applied strain `ε_applied = 2·U_MAX / L_y`.

> The fit usually overshoots `ε_applied` by a few percent. That's the
> well-known bond-based PD **surface effect**: nodes near the top/bottom
> boundaries have fewer neighbours than bulk nodes, so the material is
> effectively softer there, which the interior sees as slightly extra
> strain. PeriPy can correct this with `surface_correction=...`.

In [ ]:
coords = model.coords[:, :2]

# Pick a strip in the middle of the plate (avoid x-boundary effects)
middle_mask = np.abs(coords[:, 0] - L_x / 2.0) < 0.05 * L_x
y_mid = coords[middle_mask, 1]
uy_mid = u[middle_mask, 1]

A = np.vstack([y_mid, np.ones_like(y_mid)]).T
slope, intercept = np.linalg.lstsq(A, uy_mid, rcond=None)[0]

sigma_analytical = E * eps_applied

print(f'applied strain   eps_applied = {eps_applied:.3e}')
print(f'measured strain  du_y/dy     = {slope:.3e}')
print(f'relative error               = {100 * (slope - eps_applied) / eps_applied:+.2f} %')
print(f'analytical stress sigma      = {sigma_analytical:.3e} Pa')

## 9. Visualisation

In [ ]:
scale = 500.0  # big exaggeration - elastic displacements are tiny
deformed = coords + scale * u[:, :2]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.0))

sc0 = axes[0].scatter(coords[:, 0], coords[:, 1], c=u[:, 1],
                      cmap='coolwarm', s=6)
axes[0].set_title('u_y (undeformed)')
axes[0].set_xlabel('x [m]'); axes[0].set_ylabel('y [m]')
axes[0].set_aspect('equal')
plt.colorbar(sc0, ax=axes[0], label='u_y [m]')

sc1 = axes[1].scatter(deformed[:, 0], deformed[:, 1], c=u[:, 1],
                      cmap='coolwarm', s=6)
axes[1].set_title(f'Deformed (×{scale:.0f})')
axes[1].set_xlabel('x [m]'); axes[1].set_ylabel('y [m]')
axes[1].set_aspect('equal')
plt.colorbar(sc1, ax=axes[1], label='u_y [m]')

ys_plot = np.linspace(0, L_y, 100)
uy_lin = eps_applied * (ys_plot - L_y / 2.0)
axes[2].plot(uy_mid * 1e6, y_mid, 'o', ms=3, alpha=0.5, label='PeriPy midplane')
axes[2].plot(uy_lin * 1e6, ys_plot, 'k--', lw=1.5,
             label=f'ε·(y − L_y/2),  ε={eps_applied:.1e}')
axes[2].set_xlabel('u_y [μm]'); axes[2].set_ylabel('y [m]')
axes[2].set_title('Axial u_y profile')
axes[2].legend(loc='best'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Next steps

- **Surface correction**: pass `surface_correction='volume correction'`
  or similar to `Model(...)` and watch the fit error drop toward zero.
- **Raise the load**: increase `U_MAX` past `L_y/2 · critical_stretch`
  and you'll reproduce the fracture demo — the plate snaps.
- **Bending**: change the BC function to fix the left edge (`x < BC_LAYER`)
  and apply a force BC on the right edge via `is_force_boundary=...` to
  study an elastic cantilever.
- **Material sweep**: vary `E` and verify the fitted `du_y/dy` stays
  constant (the imposed strain is geometric, not material).
- **Mesh refinement**: double `nx`, `ny` and reduce `dt` so the surface
  effect shrinks (more neighbours per node near the boundary).